# 🎓 Ch18：Offline RL（CQL / IQL / Decision Transformer）—— 用历史数据学策略

> **Ch15 §15.6.3** 列了 7 个开放研究方向，本章展开**第 4 条**——Offline RL / Decision Transformer。
>
> **也是 Phase 4 的终章、整个 RLStudy 项目（Ch00-Ch18 共 19 章）的最终章节。**

本章的核心问题：

> **如果只有历史交互数据集 $\mathcal{D}$（不能再和环境交互），怎么学一个好策略？**

这是 **在线 RL**（Ch06 DQN / Ch09 PPO / Ch13 GRPO）无法回答的问题——它们都假设
agent 能反复 rollout。但真实场景里，反复试错常常**不可承受**：

| 场景 | 为什么不能在线试错 |
|---|---|
| **医疗** | 不能反复"试"不同治疗方案 |
| **自动驾驶** | 不能反复撞车 |
| **工业控制** | 试错代价高（机器会坏） |
| **LLM 对齐** | 在线 rollout 也要算 RM、采 trajectory，慢且贵 |

Offline RL 的回答：

> **在 Q-learning 上加保守性（CQL）、或避免 OOD 评估（IQL）、或干脆放弃 Q 改用监督学习（DT）—— 三条路都能从纯历史数据学到能用的策略。**

本章三方法都来自 Ch15 §15.6.3 第 4 条，是工业界**最实用**的 RL 落地方向之一。

### 章节路线图

| 节 | 主题 | 关键问题 |
|---|---|---|
| **18.1** | Offline RL 动机 | 为什么不能在线学？ |
| **18.2** | 核心难点：distribution shift | 为什么 naive Q-learning 会爆炸？ |
| **18.3** | CQL（保守 Q-Learning） | 怎么"压低 OOD action 的 Q"？ |
| **18.4** | IQL（隐式 Q-Learning） | 怎么"完全不评估 OOD"？ |
| **18.5** | Decision Transformer | 怎么"放弃 Q-learning 改用监督学习"？ |
| **18.6** | 三方对比实验 | 哪个方法最好？ |
| **18.7** | Phase 4 终章 + 项目最终总结 | RLStudy 全书回顾 |

> **配套**：`utils/offline_rl.py`（CQL/IQL/DT 三方法实现）+ `tests/test_offline_rl.py`（23 个冒烟测试，全部通过）


In [ ]:
# 常规设置：找项目根、载入库
import sys, pathlib, time, math, random
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ch06 基础设施（DQN 的 Q 网络、ReplayBuffer、target net 工具）
from rlenvs import CartPoleLite
from utils.networks import QNetwork, make_mlp
from utils.replay import ReplayBuffer
from utils.dqn_utils import hard_update, polyak_update
from utils import set_seed
from utils.torch_utils import get_device, count_parameters

# 本章新基础设施
from utils.offline_rl import (
    collect_offline_dataset, OfflineDataset,
    random_policy_factory, heuristic_cartpole_policy,
    offline_dqn_update_step,
    cql_loss, CQLTrainer,
    expectile_loss, IQLTrainer,
    DecisionTransformer, dt_loss, DTTrainer, dt_rollout,
    evaluate_policy,
)
from utils.viz import smooth, plot_training_curve, plot_bar_compare

set_seed(42)
torch.manual_seed(42); np.random.seed(42); random.seed(42)

DEVICE = "cpu"
print(f"PyTorch: {torch.__version__}, device = {DEVICE}")
print()
print("本章新基础设施: utils/offline_rl.py")
print("  - collect_offline_dataset        (用 policy 采离线数据集)")
print("  - OfflineDataset                 (容器：采样 + return-to-go 计算)")
print("  - offline_dqn_update_step        (naive offline DQN, 对照实验)")
print("  - cql_loss / CQLTrainer          (Conservative Q-Learning)")
print("  - expectile_loss / IQLTrainer    (Implicit Q-Learning)")
print("  - DecisionTransformer / DTTrainer (RL = Sequence Modeling)")
print("  - dt_rollout / evaluate_policy   (评估工具)")
print("  - tests/test_offline_rl.py: 23 个冒烟测试")


## 18.1 Offline RL 动机

### 18.1.1 在线 RL 的隐含假设

回顾前面所有章节的 RL 算法——它们都隐含一个**强假设**：

> **Agent 可以反复和环境交互（rollout）**

| 章节 | 算法 | 怎么用环境 |
|---|---|---|
| Ch05 | Q-learning | 在线 ε-greedy 探索 GridWorld |
| Ch06 | DQN | 在线 ε-greedy 采 → 喂 replay buffer |
| Ch07 | REINFORCE | 在线 rollout 整条 trajectory |
| Ch08 | Actor-Critic + GAE | 在线 rollout + 估 advantage |
| Ch09 | PPO | 在线 rollout → importance sampling |
| Ch12 | RLHF-PPO | 在线用 actor 生成 response |
| Ch13 | GRPO | 在线采 group of responses |

甚至 Ch06 的 **off-policy DQN** 也在线：replay buffer 只是"提高样本利用率"的辅助，
DQN 仍然需要持续 ε-greedy 探索新数据。

### 18.1.2 真实场景里"在线试错"常常不可承受

**例 1：医疗**
- 治疗一位脓毒症（sepsis）病人，每 4 小时要决定静脉输液量、升压药剂量
- 不能"试"多种方案——病人只有一个，试错了会死
- 我们能拿到的是**历史病例**：其它医生怎么治、病人活没活

**例 2：自动驾驶**
- 训练一辆车，每秒要做几百个决策
- 不能让 RL 反复撞墙学——车会坏、人会死
- 我们能拿到的是**历史驾驶 log**：人类司机怎么开

**例 3：LLM 对齐**
- RLHF（Ch12）/GRPO（Ch13）训练时，actor 在线生成 response
- 但每条 rollout 都要：actor 前向 → reward model 评分 → 算 advantage → 反传
- 百亿参数模型上，**rollout 比反向传播还贵**
- 如果有大量历史对话 log（用户 + 旧模型），能不能直接用？

### 18.1.3 Offline RL 的设定

> **Offline RL（也叫 Batch RL）**：给定一个**预先收集好的**数据集 $\mathcal{D}$
> （由某个 **behavior policy** $\pi_b$ 产生），从中学习一个策略 $\pi$，
> **整个过程不再与环境交互**。学完直接部署 $\pi$。

数学上：

- 数据集：$\mathcal{D} = \{(s_t, a_t, r_t, s_{t+1})\}$，由 $\pi_b$ 在 env 中采
- 目标：找到 $\pi^*$ 使 $\mathbb{E}_{\pi^*}[\sum_t \gamma^t r_t]$ 最大
- **约束**：训练中**不能**调 `env.step`

与前面章节的关系：

```
在线 on-policy  ←→  在线 off-policy  ←→  offline
（Ch07-09）         （Ch05-06）          （Ch18）
REINFORCE/PPO       Q-learning/DQN       CQL/IQL/DT
每步采新数据        采新数据 + 喂 buffer   完全不采新数据
```

Offline RL 是 RL "数据来源"维度上的极端：

| 维度 | on-policy | off-policy | **offline** |
|---|---|---|---|
| 能用历史数据吗？ | 否 | 是 | **是** |
| 能采新数据吗？ | 是 | 是 | **否** |
| 代表 | REINFORCE, PPO | DQN, SAC | **CQL, IQL, DT** |

> **⚠️ 注意**：Ch06 的 DQN 虽然叫 "off-policy"，但仍**在线**采。
> Offline RL 是"连 off-policy 都不能采"的更严格设定。


In [ ]:
# 18.1.4 用一个具体例子建立直觉：在 CartPoleLite 上构造 offline 数据集
#
# 我们用一个 "部分训练过的启发式" 当 behavior policy 采数据。
# 这个策略比 random 强一点（能撑住几步），但远不是最优——
# 模拟 "我们手头有一批中等质量的历史数据" 的真实场景。

env_demo = CartPoleLite(max_steps=200, seed=0)
print(f"环境: CartPoleLite, state_dim={env_demo.observation_dim}, n_actions={env_demo.nA}")
print(f"  状态 = [x, x_dot, theta, theta_dot]")
print(f"  动作 = 0 (向左推), 1 (向右推)")
print(f"  奖励 = 每步 +1.0, max_steps = {env_demo.max_steps}")
print()

# 用三个不同 behavior policy 各采 5 条 episode，对比数据质量
policies = {
    'random':    random_policy_factory(env_demo.nA, seed=1),
    'heuristic': heuristic_cartpole_policy,
}
print("Behavior policy 对比（每条采 5 episodes）:")
for name, p in policies.items():
    env_tmp = CartPoleLite(max_steps=200, seed=0)
    buf = collect_offline_dataset(env_tmp, p, n_episodes=5, seed=42)
    ep_lens = [e - s for s, e in OfflineDataset.from_buffer(buf).episode_boundaries]
    print(f"  {name:10s}: {len(buf):>5d} transitions, "
          f"avg episode length = {np.mean(ep_lens):.1f} ± {np.std(ep_lens):.1f}")

print()
print("观察：启发式策略撑住的步数比纯随机明显更长（数据集质量更高）。")
print("本章的 offline 数据集将混合这两种 policy，模拟真实场景。")


## 18.2 核心难点：distribution shift

### 18.2.1 为什么 naive Q-learning 在 offline 上会爆炸？

回顾 DQN 的 Bellman target：

$$y_t = r_t + \gamma \max_{a'} Q(s_{t+1}, a'; \theta^-)$$

那个 $\max_{a'}$ 是关键——它对**所有** $a'$（包括 $\pi_b$ 从没采过的 OOD action）评估 $Q$。

**在线 setting**：这没事，因为接下来 ε-greedy 会去访问那些 OOD action，数据会
"修正"高估的 Q。

**Offline setting**：数据集**不更新**，Q 对 OOD action 的高估**永远无法被修正**。
而且 DQN 的 $\max$ 算子有**系统性正向偏置**（max 的期望 ≥ 期望的 max）：

$$\mathbb{E}[\max_a \hat{Q}_a] \ge \max_a \mathbb{E}[\hat{Q}_a] = \max_a Q_a$$

每次 Bellman backup 都把这种正向偏置**累积放大**：

1. 初始 $Q$ 对某些 OOD action 偶然偏大
2. $\max$ 选了这些 OOD action 作为 $Q(s', a^*)$
3. $Q(s, a)$ 用这个偏大的 target 更新 → $Q(s, a)$ 也变大
4. 下一轮 $\max$ 又放大一轮

→ **指数式发散**，Q 值爆炸。

### 18.2.2 数值演示：naive offline DQN 在数据集上发散

下面我们做一个**最小化对照实验**：用同一个 offline 数据集训两个 DQN，
- 一个是标准 DQN（应该爆炸）
- 一个用我们 Ch18 的新方法（CQL，应该稳定）

先看标准 DQN 的"爆炸"行为。

> 🤔 **先猜再跑**：naive offline DQN 的 Q 值曲线会是什么形状？(A) 平稳收敛到真值附近；(B) 缓慢漂移、越学越不准；(C) 先正常学一段，然后 Q 值**指数式起飞**，几千步内冲到天文数字。
>
> <details><summary>选完再跑下面的对照</summary>
>
> 关键在 max 的小动作：网络对**从没见过的动作**（OOD）输出的 Q 是随机的——而 max 恰好专挑"碰巧虚高"的那个。在线时 ε-greedy 会真的去试它、数据把虚高压下去；offline 时数据集是死的，虚高被 target 反复放大——每一步都拿"最大幻觉"当目标。这不是学得慢，是**自说自话的复利**。猜 (C) 的读者，你已经在用 offline RL 研究者的眼睛看问题了。
> </details>


In [ ]:
# 18.2.3 构造 Ch18 全章使用的 offline 数据集
#
# 设计：混合 random + heuristic policy 各采若干 episode，让数据集
# - 既覆盖大量 (s, a) 组合（random 的贡献）
# - 又有一些"撑住几步"的高 quality 轨迹（heuristic 的贡献）
# 模拟真实场景：日志里既有"瞎试"也有"靠谱操作"。

N_EPISODES_RANDOM = 400
N_EPISODES_HEURISTIC = 400
ENV_MAX_STEPS = 200

env_data = CartPoleLite(max_steps=ENV_MAX_STEPS, seed=0)
t0 = time.time()
# 用 random policy 采
buf_random = collect_offline_dataset(
    env_data, random_policy_factory(env_data.nA, seed=1),
    n_episodes=N_EPISODES_RANDOM, seed=100,
)
# 用 heuristic policy 采
buf_heur = collect_offline_dataset(
    env_data, heuristic_cartpole_policy,
    n_episodes=N_EPISODES_HEURISTIC, seed=200,
)

# 合并两个 buffer 成一个大数据集
all_states = np.concatenate([buf_random._states[:buf_random._size],
                             buf_heur._states[:buf_heur._size]])
all_actions = np.concatenate([buf_random._actions[:buf_random._size],
                              buf_heur._actions[:buf_heur._size]])
all_rewards = np.concatenate([buf_random._rewards[:buf_random._size],
                              buf_heur._rewards[:buf_heur._size]])
all_next = np.concatenate([buf_random._next_states[:buf_random._size],
                           buf_heur._next_states[:buf_heur._size]])
all_dones = np.concatenate([buf_random._dones[:buf_random._size],
                            buf_heur._dones[:buf_heur._size]])

# 打包成 OfflineDataset（按 done 切 episode）
N = len(all_states)
boundaries = []
start = 0
for i in range(N):
    if all_dones[i] >= 0.5:
        boundaries.append((start, i + 1))
        start = i + 1
if start < N:
    boundaries.append((start, N))

DATASET = OfflineDataset(all_states, all_actions, all_rewards,
                          all_next, all_dones, boundaries)
DATA_TIME = time.time() - t0

ep_lens = np.array([e - s for s, e in boundaries])
print(f"Offline dataset 构造完成 ({DATA_TIME:.1f}s):")
print(f"  Total transitions: {DATASET.n}")
print(f"  Total episodes:    {len(boundaries)}")
print(f"  Episode length:    mean={ep_lens.mean():.1f}, "
      f"std={ep_lens.std():.1f}, max={ep_lens.max()}, min={ep_lens.min()}")
print(f"  State dim:         {DATASET.state_dim}")
print(f"  Action distribution: 0={float((DATASET.actions==0).mean()):.2%}, "
      f"1={float((DATASET.actions==1).mean()):.2%}")
print()
print("这就是 Ch18 全章使用的固定数据集。下面三个方法（CQL/IQL/DT）都只读它，不再调 env。")


In [ ]:
# 18.2.4 数据集可视化：behavior policy 的 (state, action) 覆盖
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 左图：state 分布（用 theta vs theta_dot 投影）
ax = axes[0]
theta = DATASET.states[:, 2]
theta_dot = DATASET.states[:, 3]
# 区分 action 0 / 1
a0 = DATASET.actions == 0
a1 = DATASET.actions == 1
ax.scatter(theta[a0], theta_dot[a0], s=4, alpha=0.3, c='#1f77b4', label='action=0 (left)')
ax.scatter(theta[a1], theta_dot[a1], s=4, alpha=0.3, c='#ff7f0e', label='action=1 (right)')
ax.axvline(-env_data.theta_threshold, color='r', ls='--', alpha=0.5, label='termination')
ax.axvline(env_data.theta_threshold, color='r', ls='--', alpha=0.5)
ax.set_xlabel('theta (rad)')
ax.set_ylabel('theta_dot (rad/s)')
ax.set_title('Offline dataset 的 state-action 覆盖\n(只覆盖了 π_b 访问过的区域)')
ax.legend(markerscale=4, fontsize=8, loc='upper right')

# 右图：episode length 直方图
ax = axes[1]
ax.hist(ep_lens, bins=30, color='steelblue', alpha=0.7, edgecolor='k')
ax.axvline(ep_lens.mean(), color='crimson', linewidth=2,
           label=f'mean = {ep_lens.mean():.1f}')
ax.set_xlabel('Episode length (steps)')
ax.set_ylabel('Count')
ax.set_title('Episode 长度分布\n(混合 random + heuristic policy)')
ax.legend()

plt.tight_layout(); plt.show()
print(f"覆盖范围: theta ∈ [{theta.min():.3f}, {theta.max():.3f}] rad "
      f"(±{env_data.theta_threshold:.3f} 终止)")
print(f"如果学到的 π 去了 θ > 0.2 的区域（图中稀疏处），Q 估计就不可靠（外推）。")


In [ ]:
# 18.2.5 Naive offline DQN 实验：让标准 DQN 在固定数据集上学，看 Q 值爆炸
#
# 这是 §18.2 理论预测的"灾难"实验：在纯 offline 数据上跑标准 DQN，
# Q 值应该指数式发散。下面我们重现这个失败。

torch.manual_seed(42); np.random.seed(42)
NAIVE_DQN_STEPS = 1500
naive_q = QNetwork(DATASET.state_dim, 2, [128, 128])
naive_tgt = QNetwork(DATASET.state_dim, 2, [128, 128])
hard_update(naive_tgt, naive_q)
naive_opt = torch.optim.Adam(naive_q.parameters(), lr=1e-3)

naive_history = []
t0 = time.time()
for step in range(NAIVE_DQN_STEPS):
    batch = DATASET.sample(64, rng=np.random.default_rng(step))
    stats = offline_dqn_update_step(naive_q, naive_tgt, naive_opt, batch, gamma=0.99)
    # hard target update every 100 steps
    if (step + 1) % 100 == 0:
        hard_update(naive_tgt, naive_q)
    if step % 10 == 0:
        naive_history.append({"step": step, **stats})

NAIVE_TIME = time.time() - t0

# 评估
naive_eval = evaluate_policy(
    lambda s: int(naive_q(torch.as_tensor(s, dtype=torch.float32).unsqueeze(0))
                  .argmax(dim=1)[0].item()),
    CartPoleLite(max_steps=200, seed=999), n_episodes=10, seed=1234,
)
print(f"Naive offline DQN ({NAIVE_DQN_STEPS} steps, {NAIVE_TIME:.1f}s):")
print(f"  Final loss     = {naive_history[-1]['loss']:.4f}")
print(f"  Final Q mean   = {naive_history[-1]['q_mean']:.4f}")
print(f"  Final Q max    = {naive_history[-1]['q_max']:.4f}")
print(f"  Eval reward    = {naive_eval['mean']:.1f} ± {naive_eval['std']:.1f}")
print()
q_means = [h['q_mean'] for h in naive_history]
q_maxs = [h['q_max'] for h in naive_history]
print(f"Q mean 从 {q_means[0]:.2f} → {q_means[-1]:.2f} "
      f"(增长倍数 = {q_means[-1] / max(q_means[0], 1e-3):.1f}x)")
print(f"Q max  从 {q_maxs[0]:.2f} → {q_maxs[-1]:.2f} "
      f"(增长倍数 = {q_maxs[-1] / max(q_maxs[0], 1e-3):.1f}x)")


In [ ]:
# 18.2.6 可视化：Q 值发散
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

steps = [h['step'] for h in naive_history]
ax = axes[0]
ax.plot(steps, [h['q_mean'] for h in naive_history], linewidth=2, label='Q mean (in-dist)')
ax.plot(steps, [h['q_max'] for h in naive_history], linewidth=2, label='Q max (incl. OOD)')
ax.set_xlabel('Training step')
ax.set_ylabel('Q value')
ax.set_title('Naive offline DQN: Q 值发散\n(理论上应爆炸——max 累积正向偏置)')
ax.legend()

# 真实可达的最大 Q（用 heuristic policy 能撑住 ~50 步，γ=0.99 → 真实 Q ≈ 50）
true_max_q = ep_lens.mean() / (1 - 0.99)  # 几何级数和的近似
ax.axhline(true_max_q, color='green', linewidth=2, linestyle='--',
           label=f'true max Q ≈ {true_max_q:.0f} (数据集 avg return / (1-γ))')
ax.legend()
ax.set_ylim(None, max(q_maxs) * 1.1)

# 右图：Q 值在 state 空间的分布（用 theta 投影）
ax = axes[1]
sample_idx = np.random.choice(DATASET.n, 200, replace=False)
sample_states = DATASET.states[sample_idx]
with torch.no_grad():
    q_vals = naive_q(torch.as_tensor(sample_states, dtype=torch.float32)).numpy()
theta_samples = sample_states[:, 2]
ax.scatter(theta_samples, q_vals[:, 0], s=20, alpha=0.6, c='#1f77b4', label='Q(s, 0)')
ax.scatter(theta_samples, q_vals[:, 1], s=20, alpha=0.6, c='#ff7f0e', label='Q(s, 1)')
ax.set_xlabel('theta (rad)')
ax.set_ylabel('Q value')
ax.set_title(f'Q 在 state 空间的分布\n(发散后的 Q 远超真实可达范围)')
ax.legend()

plt.tight_layout(); plt.show()
print(f"理论上 Q(s, a) 应该 ≤ ~{true_max_q:.0f}（数据集 avg return 的几何级数和）。")
print(f"实际 naive DQN 把 Q 推到 {q_maxs[-1]:.0f}，远超真实值——这就是外推错误。")


### 18.2.7 三个解法方向（本章剩余三节展开）

| 方法 | 思路 | 出处 |
|---|---|---|
| **CQL**（§18.3） | Q-learning + 保守正则：压低 OOD action 的 Q | Kumar et al. 2020 |
| **IQL**（§18.4） | 完全避免 OOD action 评估（用 V 替代 max） | Kostrikov et al. 2022 |
| **DT**（§18.5） | 放弃 Q-learning，把 RL 重写成监督学习 | Chen et al. 2021 |

还有第四条路：**TD3+BC**（Fujimoto & Gu 2021）——在 BC（行为克隆）上加一点 RL，
思路是"主要模仿数据集，RL 只是微调"。本章不展开，但会在 §18.6 对比里提到。


## 18.3 CQL: Conservative Q-Learning

### 18.3.1 思路

CQL（Kumar et al. 2020, NeurIPS）的回答：

> **仍然做 $\max_a Q$（保留 Q-learning），但加一个正则把 OOD action 的 Q 压低，
> 让 $\arg\max_a Q$ 倾向于选 in-distribution action。**

具体来说，在标准 DQN loss 上加 **CQL regularizer**：

$$
\boxed{\;
\mathcal{L}_{CQL}(\theta) =
\underbrace{\mathcal{L}_{DQN}(\theta)}_{\text{Bellman 残差}}
+ \alpha \cdot \underbrace{\left[
  \mathbb{E}_{s \sim \mathcal{D}}\!\left[\log \sum_a \exp(Q(s, a; \theta))\right]
  - \mathbb{E}_{(s, a) \sim \mathcal{D}}[Q(s, a; \theta)]
\right]}_{\text{CQL regularizer}}
\;}
$$

两项的物理含义：

- **第一项 $\log \sum_a \exp(Q(s, a))$**：log-sum-exp（光滑 soft-max），是 $\max_a Q(s, a)$ 的**光滑上界**。**压低它**就压低了所有 $a$（特别是 OOD）的 Q 上界。
- **第二项 $-\mathbb{E}_{(s,a) \sim \mathcal{D}}[Q(s, a)]$**：让**数据集内** $(s, a)$ 的 Q **不被压低**（注意符号是负的）。

组合起来：**OOD action 的 Q 被压低，in-distribution action 的 Q 维持**——
$\arg\max_a Q$ 自然倾向选 in-distribution action。

### 18.3.2 为什么用 log-sum-exp 而不是 max？

数学上 $\log \sum_a e^{Q_a}$ 满足：

$$\max_a Q_a \;\le\; \log \sum_a e^{Q_a} \;\le\; \max_a Q_a + \log|A|$$

- 是 $\max_a Q$ 的**紧上界**（差 $\le \log|A|$，CartPole 的 $|A|=2$ 时差 $\le 0.69$）
- $\max$ 只对 argmax 那个 $a$ 有梯度，log-sum-exp **对所有 $a$ 都有梯度**——
  训练更稳定（梯度信号密集）

### 18.3.3 CQL regularizer 的两项目标

```
minimize L_CQL(θ) =
    minimize L_DQN(θ)              ← 让 Q 满足 Bellman 方程
    +
    α · minimize logsumexp(Q)      ← 压低所有 action 的 Q 上界（特别是 OOD）
    α · maximize Q(s_data, a_data) ← 维持数据集内 (s, a) 的 Q
```

最终效果：**OOD action 的 Q 被压低，in-distribution action 的 Q 维持高**。

> 直觉：CQL 学到的是 Q 的一个**保守下界**——真实 $Q^*$ 可能更高，但 CQL 估的
> 不会因外推而偏高，所以 $\arg\max_a Q_{CQL}$ 不会选到 OOD 的"虚假最优"。

<details>
<summary><b>📐 CQL 推导细节：从分布鲁棒优化出发</b></summary>

CQL 的原始推导（Kumar 2020）从**分布鲁棒优化**（distributionally robust optimization）
出发：

考虑真实 $Q^*$ 是某个未知分布的期望，而我们只有数据集 $\mathcal{D}$。
对 OOD 区域的 $Q$ 没有数据约束，所以最坏情况（worst-case）下 $Q$ 可以任意大。
CQL 显式地**最小化这个最坏情况的上界**：

$$\min_\theta \max_{\mu \in \mathcal{U}} \;
\mathbb{E}_{s \sim \mathcal{D}, a \sim \mu}[Q_\theta(s, a)]
- \mathbb{E}_{(s, a) \sim \mathcal{D}}[Q_\theta(s, a)]$$

其中 $\mathcal{U}$ 是允许的"对抗分布"（adversarial distribution）集合。
当 $\mathcal{U}$ 是所有概率分布时，内层 $\max_\mu \mathbb{E}_{a \sim \mu}[Q]$
的解是 $\mu^* = \delta_{\arg\max_a Q}$，即点集中在 max 上 → 退化为 $\max_a Q$。

为了让 $\max$ 可微，Kumar 用 **log-sum-exp** 替代：

$$\max_a Q_a \approx \log \sum_a e^{Q_a}$$

这就是 CQL regularizer 第一项的来源。

**α 的选择**：α 控制"保守程度"。α 太小 → 退化为 DQN，仍外推；α 太大 →
过度保守，Q 被压得过低，$\arg\max$ 也无法区分 in-dist action 的优劣。
Kumar 2020 经验值：α ∈ [0.1, 10]，本章用 α = 5.0。

</details>

### 18.3.4 数值实验：CQL vs Naive DQN


In [ ]:
# 18.3.5 CQL 实验：在同样的 offline 数据集上训，应该稳定（不发散）
torch.manual_seed(42); np.random.seed(42)
CQL_STEPS = 1500
CQL_ALPHA = 5.0

cql_q = QNetwork(DATASET.state_dim, 2, [128, 128])
cql_tgt = QNetwork(DATASET.state_dim, 2, [128, 128])
cql_opt = torch.optim.Adam(cql_q.parameters(), lr=1e-3)
cql_trainer = CQLTrainer(
    cql_q, cql_tgt, cql_opt, DATASET,
    n_actions=2, alpha=CQL_ALPHA, gamma=0.99,
    tau=0.005, batch_size=64,
)

cql_history = []
t0 = time.time()
for step in range(CQL_STEPS):
    stats = cql_trainer.step()
    if step % 10 == 0:
        cql_history.append({"step": step, **stats})
CQL_TIME = time.time() - t0

# 评估
cql_eval = evaluate_policy(
    lambda s: cql_trainer.greedy_action(s),
    CartPoleLite(max_steps=200, seed=999), n_episodes=10, seed=1234,
)
print(f"CQL ({CQL_STEPS} steps, α={CQL_ALPHA}, {CQL_TIME:.1f}s):")
print(f"  Final bellman_loss = {cql_history[-1]['bellman_loss']:.4f}")
print(f"  Final cql_reg      = {cql_history[-1]['cql_reg']:.4f}")
print(f"  Final Q mean (in-dist) = {cql_history[-1]['q_mean']:.4f}")
print(f"  Final Q max  (incl OOD) = {cql_history[-1]['q_all_max']:.4f}")
print(f"  Final lse          = {cql_history[-1]['lse']:.4f}")
print(f"  Eval reward        = {cql_eval['mean']:.1f} ± {cql_eval['std']:.1f}")
print()
print(f"对比 naive DQN: Q max = {naive_history[-1]['q_max']:.2f}")
print(f"          CQL : Q max = {cql_history[-1]['q_all_max']:.2f}")
print(f"CQL 的 Q max 明显比 naive 小——保守正则把 OOD Q 压低了。")


In [ ]:
# 18.3.6 CQL vs Naive DQN: Q 值发散对比（关键图）
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
steps_n = [h['step'] for h in naive_history]
steps_c = [h['step'] for h in cql_history]
ax.plot(steps_n, [h['q_max'] for h in naive_history], linewidth=2,
        color='#d62728', label='Naive DQN: Q max (incl OOD)')
ax.plot(steps_c, [h['q_all_max'] for h in cql_history], linewidth=2,
        color='#2ca02c', label=f'CQL (α={CQL_ALPHA}): Q max (incl OOD)')
true_max_q = ep_lens.mean() / (1 - 0.99)
ax.axhline(true_max_q, color='gray', linestyle='--', linewidth=1.5,
           label=f'true max Q ≈ {true_max_q:.0f}')
ax.set_xlabel('Training step')
ax.set_ylabel('Q max (over all actions, including OOD)')
ax.set_title('CQL 把 OOD Q 压低（vs Naive DQN 发散）')
ax.legend(fontsize=9)
ax.set_ylim(0, max(max(h['q_max'] for h in naive_history),
                   max(h['q_all_max'] for h in cql_history)) * 1.1)

# 右图：bellman loss 对比（CQL 应该有合理的 bellman loss，naive 也合理但 Q 发散）
ax = axes[1]
ax.plot(steps_n, [h['loss'] for h in naive_history], linewidth=2,
        color='#d62728', label='Naive DQN: Bellman loss')
ax.plot(steps_c, [h['bellman_loss'] for h in cql_history], linewidth=2,
        color='#2ca02c', label='CQL: Bellman loss (component)')
ax.set_xlabel('Training step')
ax.set_ylabel('Bellman MSE loss')
ax.set_title('Bellman loss 都收敛，但 Naive 的 Q 仍爆炸（max 的偏置不在 loss 里体现）')
ax.legend(fontsize=9)
ax.set_ylim(0, max(max(h['loss'] for h in naive_history),
                   max(h['bellman_loss'] for h in cql_history)) * 1.2)

plt.tight_layout(); plt.show()
print("关键洞察：Bellman loss 看不出 Q 发散——naive DQN 的 loss 也在降。")
print("灾难发生在 max 的偏置上——只能通过监控 Q max 和评估 reward 发现。")
print(f"CQL 通过 cql_reg 把 Q max 压在合理范围内，所以学出的策略能正常工作。")


In [ ]:
# 18.3.7 CQL 学到的策略 vs Naive DQN 学到的策略：可视化 state 空间
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 在 state 空间网格上算 Q
theta_grid = np.linspace(-0.21, 0.21, 30)
theta_dot_grid = np.linspace(-3, 3, 30)
T, TD = np.meshgrid(theta_grid, theta_dot_grid)
states_grid = np.zeros((T.size, 4))
states_grid[:, 2] = T.flatten()
states_grid[:, 3] = TD.flatten()
states_t = torch.as_tensor(states_grid, dtype=torch.float32)

with torch.no_grad():
    q_naive = naive_q(states_t).numpy()
    q_cql = cql_q(states_t).numpy()

# 选 argmax_a Q 的 action
ax = axes[0]
best_a_naive = q_naive.argmax(axis=1).reshape(T.shape)
c = ax.contourf(T, TD, best_a_naive, levels=[-0.5, 0.5, 1.5],
                colors=['#1f77b4', '#ff7f0e'], alpha=0.7)
ax.scatter(DATASET.states[:, 2], DATASET.states[:, 3], s=1, alpha=0.15, c='k')
ax.set_xlabel('theta'); ax.set_ylabel('theta_dot')
ax.set_title('Naive DQN: argmax_a Q(s, a)\n(可能选 OOD 高 Q action → 不合理策略)')

ax = axes[1]
best_a_cql = q_cql.argmax(axis=1).reshape(T.shape)
ax.contourf(T, TD, best_a_cql, levels=[-0.5, 0.5, 1.5],
            colors=['#1f77b4', '#ff7f0e'], alpha=0.7)
ax.scatter(DATASET.states[:, 2], DATASET.states[:, 3], s=1, alpha=0.15, c='k')
ax.set_xlabel('theta'); ax.set_ylabel('theta_dot')
ax.set_title(f'CQL (α={CQL_ALPHA}): argmax_a Q(s, a)\n(应近似 PD 控制器: theta > 0 → 右推)')

plt.tight_layout(); plt.show()
print("正确策略（PD 控制器）：theta > 0 → 向右推 (action=1)，theta < 0 → 向左推 (action=0)。")
print(f"CQL eval reward: {cql_eval['mean']:.1f}, naive eval: {naive_eval['mean']:.1f}")


## 18.4 IQL: Implicit Q-Learning

### 18.4.1 思路

IQL（Kostrikov et al. 2022, ICLR）的回答更激进：

> **完全不评估 OOD action 的 Q。** 用一个 $V$ 网络替代 Bellman target 里的 $\max_a Q$，
> 而 $V$ 只用**数据集内的 $(s, a)$** 学——彻底绕开外推。

关键工具：**Expectile Regression**。

### 18.4.2 Expectile 概念回顾

Expectile $\tau \in (0, 1)$ 是 minimize 下面这个 asymmetric squared loss 的解：

$$e_\tau(X) = \arg\min_e \mathbb{E}\big[ \rho_\tau(X - e) \big]$$

$$\rho_\tau(u) = |\tau - \mathbb{1}(u < 0)| \cdot u^2$$

性质：

- $\tau = 0.5$：退化为 mean（标准 MSE）
- $\tau > 0.5$：偏向上尾（残差正时权重大 → $e$ 被往上拉）
- $\tau \to 1$：退化为 max

```
                  τ=0.5    τ=0.7    τ=0.9    τ=1.0
                  mean     ↑        ↑↑       max
                   ↓        ↓        ↓        ↓
                   ─────────●────────●────────●──→  上尾偏置增加
```


In [ ]:
# 18.4.3 Expectile 概念数值演示
# 在一个简单的非对称分布上算不同 τ 的 expectile

np.random.seed(0)
# 构造一个右偏分布：80% 在 0-1，20% 在 5-6（模拟 "好 action 的 Q 高"）
samples = np.concatenate([
    np.random.uniform(0, 1, 800),
    np.random.uniform(5, 6, 200),
])

# 用 PyTorch 优化器算 expectile
def compute_expectile(samples, tau, n_iter=500):
    s = torch.tensor(samples, dtype=torch.float32)
    e = torch.tensor([samples.mean()], requires_grad=True)
    opt = torch.optim.Adam([e], lr=0.05)
    for _ in range(n_iter):
        loss = expectile_loss(e.expand_as(s), s, expectile=tau)
        opt.zero_grad(); loss.backward(); opt.step()
    return float(e.item())

taus = [0.5, 0.7, 0.8, 0.9, 0.95, 1.0]
print("Expectile 数值演示（数据: 80% 在 [0,1], 20% 在 [5,6]）:")
print(f"  真实 mean = {samples.mean():.3f}, max = {samples.max():.3f}")
print(f"  {'tau':<8} {'expectile':<12} {'解释'}")
print(f"  {'-'*50}")
for tau in taus:
    e = compute_expectile(samples, tau)
    if tau == 0.5:
        note = "= mean"
    elif tau == 1.0:
        note = "= max"
    else:
        note = "向上尾偏"
    print(f"  τ={tau:<5.2f}  {e:<10.3f}   {note}")

# 可视化
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(samples, bins=50, color='lightgray', edgecolor='k', alpha=0.7)
for tau, color in zip([0.5, 0.7, 0.9, 1.0],
                       ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']):
    e = compute_expectile(samples, tau)
    ax.axvline(e, color=color, linewidth=2, linestyle='--',
               label=f'τ={tau} → e = {e:.2f}')
ax.set_xlabel('value')
ax.set_ylabel('count')
ax.set_title('Expectile：τ > 0.5 偏向上尾\n(IQL 用 τ=0.7~0.9 让 V(s) ≈ max_a Q 但不被 OOD 拉跑)')
ax.legend()
plt.tight_layout(); plt.show()


### 18.4.4 IQL 三步流程

IQL 的完整 pipeline：

**Step 1: 训 $V$ 网络（用 expectile regression）**

$$\mathcal{L}_V(\psi) = \mathbb{E}_{(s, a) \sim \mathcal{D}}\!\left[\rho_\tau\!\big(Q_{\bar\theta}(s, a) - V_\psi(s)\big)\right]$$

- 监督信号是 $Q_{\bar\theta}(s, a)$（**数据集内的 $a$，不评估 OOD**）
- $\tau > 0.5$ 让 $V(s)$ 学到 $Q$ 的**乐观近似**：偏向上尾（$\approx \max_a Q$），
  但因为只用 in-dist $a$，**不受 OOD 影响**

**Step 2: 训 $Q$ 网络（用 $V$ 替代 max）**

$$\mathcal{L}_Q(\theta) = \mathbb{E}\!\left[\big(r + \gamma V_\psi(s') - Q_\theta(s, a)\big)^2\right]$$

- 关键：target 里**没有 $\max_a$**，用 $V_\psi(s')$ 替代
- $V$ 已经是 $\max_a Q$ 的乐观近似，所以这个 target 接近标准 Bellman，但不外推

**Step 3: 策略提取（advantage-weighted regression）**

$$\pi(a|s) \propto \exp\!\big(\beta \cdot (Q_\theta(s, a) - V_\psi(s))\big)$$

- advantage 大的 action 概率高（与 $V$ 的近似 max 一致）
- 本章教学简化：直接用 $\arg\max_a Q$（性能略差，但代码简洁）

### 18.4.5 IQL vs CQL

| 维度 | CQL | IQL |
|---|---|---|
| **是否评估 OOD Q** | 是（但被保守化） | **否** |
| **正则项** | log-sum-exp penalty | 无（用 expectile 替代 max） |
| **额外网络** | 仅 Q + target | **V 网络 + Q + target** |
| **超参** | α（保守强度） | τ（expectile）、β（策略温度） |
| **思想** | "保守地外推" | "完全不外推" |
| **效果** | 都比 naive DQN 强很多，两者不相上下 | |


In [ ]:
# 18.4.6 IQL 实验
torch.manual_seed(42); np.random.seed(42)
IQL_STEPS = 1500
IQL_EXPECTILE = 0.7

iql_trainer = IQLTrainer(
    state_dim=DATASET.state_dim, n_actions=2, dataset=DATASET,
    hidden_dims=[128, 128], gamma=0.99,
    expectile=IQL_EXPECTILE, beta=3.0,
    tau=0.005, lr=1e-3, batch_size=64,
)

iql_history = []
t0 = time.time()
for step in range(IQL_STEPS):
    stats = iql_trainer.step()
    if step % 10 == 0:
        iql_history.append({"step": step, **stats})
IQL_TIME = time.time() - t0

iql_eval = evaluate_policy(
    lambda s: iql_trainer.greedy_action(s),
    CartPoleLite(max_steps=200, seed=999), n_episodes=10, seed=1234,
)
print(f"IQL ({IQL_STEPS} steps, τ={IQL_EXPECTILE}, {IQL_TIME:.1f}s):")
print(f"  Final V loss  = {iql_history[-1]['v_loss']:.4f}")
print(f"  Final Q loss  = {iql_history[-1]['q_loss']:.4f}")
print(f"  Final V mean  = {iql_history[-1]['v_mean']:.4f}")
print(f"  Final Q mean  = {iql_history[-1]['q_mean']:.4f}")
print(f"  Eval reward   = {iql_eval['mean']:.1f} ± {iql_eval['std']:.1f}")
print()
print(f"对比：CQL eval = {cql_eval['mean']:.1f}, IQL eval = {iql_eval['mean']:.1f}")


In [ ]:
# 18.4.7 IQL 训练曲线（V loss, Q loss, V vs Q）
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

steps = [h['step'] for h in iql_history]
ax = axes[0]
ax.plot(steps, [h['v_loss'] for h in iql_history], linewidth=2, color='#9467bd', label='V expectile loss')
ax.set_xlabel('step'); ax.set_ylabel('V loss')
ax.set_title(f'V 网络 loss (expectile, τ={IQL_EXPECTILE})')
ax.legend()

ax = axes[1]
ax.plot(steps, [h['q_loss'] for h in iql_history], linewidth=2, color='#1f77b4', label='Q Bellman loss')
ax.set_xlabel('step'); ax.set_ylabel('Q loss')
ax.set_title('Q 网络 loss (Bellman, target 用 V 而非 max)')
ax.legend()

ax = axes[2]
ax.plot(steps, [h['v_mean'] for h in iql_history], linewidth=2, color='#9467bd', label='V mean')
ax.plot(steps, [h['q_mean'] for h in iql_history], linewidth=2, color='#1f77b4', label='Q mean (in-dist)')
ax.axhline(true_max_q, color='gray', linestyle='--', linewidth=1.5,
           label=f'true max ≈ {true_max_q:.0f}')
ax.set_xlabel('step'); ax.set_ylabel('value')
ax.set_title('V 应近似 max_a Q（上尾），Q 维持 in-dist')
ax.legend(fontsize=8)

plt.tight_layout(); plt.show()
print(f"IQL 的 V 应该比 Q 略高（V 学上尾 ≈ max_a Q），但都比 naive DQN 的发散值小很多。")
print(f"V mean = {iql_history[-1]['v_mean']:.2f}, Q mean = {iql_history[-1]['q_mean']:.2f}")


## 18.5 Decision Transformer：RL = Sequence Modeling

### 18.5.1 思路：放弃 Q-learning

DT（Chen et al. 2021, NeurIPS）的回答最激进：

> **彻底放弃 Q-learning。把 RL 看成 "given target return, 生成能达到的 action" 的
> 监督学习问题。**

把 RL 转化为 **条件序列生成**（conditional sequence modeling）：

- 输入序列：$(R_1, s_1, a_1, R_2, s_2, a_2, \dots)$
  - $R_t = \sum_{t' \ge t} \gamma^{t'-t} r_{t'}$ 是 **return-to-go**（"还剩多少 reward 要拿"）
- 训练目标：给定 $(R_t, s_t)$ 预测 $a_t$（**监督 cross-entropy**）
- 推理：给大目标 $R^*$，让模型生成 $a_t$；执行后更新 $R_{t+1} = R_t - r_{t+1}$

### 18.5.2 训练 loss

$$\mathcal{L}_{DT}(\theta) = -\mathbb{E}_{(R, s, a) \sim \mathcal{D}}[\log \pi_\theta(a | R, s)]$$

**注意**：没有 Bellman equation、没有 Q、没有 $\max$、没有外推问题。
就是个监督学习分类问题（输入 $R+s$，输出 $a$ 的 logits）。

### 18.5.3 推理流程（关键）

```python
R = target_return  # 比如 200（希望撑 200 步）
s = env.reset()
for t in range(max_steps):
    a = model.predict(R, s)        # 给 (R, s) 预测 a
    s_next, r, done = env.step(a)
    R = R - r                       # return-to-go 递减
    s = s_next
```

**关键洞察**：DT 学的不是 "如何最大化 return"，而是 "**历史上达到 return R 时采取了什么 action**"。

- 条件化在大 $R$ → 模型生成 "历史上高 return trajectory 里的 action"
- 前提：**数据集里有高 return 轨迹**（否则模型没见过，会输出垃圾）
- 经验上：只要数据集覆盖够好，DT 的策略接近最优

### 18.5.4 与 Q-learning 的根本区别

| 维度 | Q-learning（CQL/IQL） | DT |
|---|---|---|
| **目标** | 学 $Q(s, a)$ | 学 $\pi(a \| R, s)$ |
| **推理** | $\arg\max_a Q$ | 给大 $R$，模型生成 $a$ |
| **Bellman？** | 是（核心） | **否**（完全监督） |
| **外推问题？** | 有（CQL/IQL 各有解法） | **无**（不评估任何 OOD Q） |
| **数据集要求** | 任何数据集都行 | **需要含高 return 轨迹** |
| **能"超越"数据集？** | 可以（RL 能学新策略） | **不能**（只会模仿历史） |

> 这是 DT 的根本限制：它**只能复制数据集中的成功行为**，不能像 RL 那样"组合"出新策略。
> 但实践中只要数据集够好，"复制最优历史行为"已经够用。


In [ ]:
# 18.5.5 Decision Transformer 实验
#
# 本章的 DT 是教学简化版：用 MLP（而非 Transformer），输入 = [R_t, s_t] 拼接，
# 输出 = action logits。这样可以聚焦于 "return-conditioning" 思想，不被
# Transformer 细节淹没（§18.5.6 会讨论真实 DT 与本简化的区别）。

torch.manual_seed(42); np.random.seed(42)
DT_STEPS = 1500

dt_model = DecisionTransformer(
    state_dim=DATASET.state_dim, n_actions=2, hidden_dims=[128, 128],
)
dt_opt = torch.optim.Adam(dt_model.parameters(), lr=1e-3)
dt_trainer = DTTrainer(dt_model, dt_opt, DATASET, batch_size=64)

dt_history = []
t0 = time.time()
for step in range(DT_STEPS):
    stats = dt_trainer.step()
    if step % 10 == 0:
        dt_history.append({"step": step, **stats})
DT_TRAIN_TIME = time.time() - t0

# 用不同 target_return 做 inference
DT_TARGETS = [50, 100, 150, 200]
dt_evals = {}
t0 = time.time()
for target in DT_TARGETS:
    rewards_per_ep = []
    for ep_seed in range(10):
        env_tmp = CartPoleLite(max_steps=200, seed=1000 + ep_seed)
        total, _ = dt_rollout(dt_model, env_tmp, target_return=float(target),
                              max_steps=200, greedy=True)
        rewards_per_ep.append(total)
    dt_evals[target] = float(np.mean(rewards_per_ep))
DT_EVAL_TIME = time.time() - t0

# 用最优 target（最高 eval）做最终评估
best_target = max(DT_TARGETS, key=lambda t: dt_evals[t])
dt_final_rewards = []
for ep_seed in range(10):
    env_tmp = CartPoleLite(max_steps=200, seed=999)
    total, _ = dt_rollout(dt_model, env_tmp, target_return=float(best_target),
                          max_steps=200, greedy=True)
    dt_final_rewards.append(total)
dt_eval = {"mean": float(np.mean(dt_final_rewards)),
           "std": float(np.std(dt_final_rewards))}

print(f"DT ({DT_STEPS} steps, {DT_TRAIN_TIME:.1f}s train + {DT_EVAL_TIME:.1f}s eval):")
print(f"  Final loss       = {dt_history[-1]['loss']:.4f}")
print(f"  Final action_acc = {dt_history[-1]['action_acc']:.4f}")
print(f"\n不同 target return 的 eval reward:")
for tgt, r in dt_evals.items():
    marker = '  *' if tgt == best_target else ''
    print(f"  target R = {tgt:>4d} → eval reward = {r:.1f}{marker}")
print(f"\n用最优 target (R={best_target}) eval: {dt_eval['mean']:.1f} ± {dt_eval['std']:.1f}")


In [ ]:
# 18.5.6 DT 训练曲线 + 不同 target return 的对比
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
steps = [h['step'] for h in dt_history]
ax.plot(steps, [h['loss'] for h in dt_history], linewidth=2, color='#1f77b4', label='CE loss')
ax.set_xlabel('step'); ax.set_ylabel('cross-entropy loss')
ax.set_title('DT 训练 loss（监督 cross-entropy，无 Bellman）')
ax.legend()
ax_twin = ax.twinx()
ax_twin.plot(steps, [h['action_acc'] for h in dt_history], linewidth=2,
             color='#d62728', label='action accuracy')
ax_twin.set_ylabel('action accuracy', color='#d62728')
ax_twin.tick_params(axis='y', labelcolor='#d62728')
ax_twin.set_ylim(0, 1)

# 右图：target return → eval reward
ax = axes[1]
ax.bar([str(t) for t in DT_TARGETS],
       [dt_evals[t] for t in DT_TARGETS],
       color=['#9ec5e8' if t != best_target else '#1f77b4' for t in DT_TARGETS])
ax.axhline(ep_lens.mean(), color='gray', linestyle='--', linewidth=1.5,
           label=f'数据集 avg episode length = {ep_lens.mean():.1f}')
ax.axhline(ep_lens.max(), color='crimson', linestyle='--', linewidth=1.5,
           label=f'数据集 max episode length = {ep_lens.max()}')
ax.set_xlabel('Target return R*')
ax.set_ylabel('Eval reward (avg over 10 eps)')
ax.set_title('DT 推理：target return 越大 → eval reward 越高？\n(理论：DT 学的是 "R 时采取什么 a")')
ax.legend(fontsize=8)

plt.tight_layout(); plt.show()
print(f"观察：")
print(f"  数据集 avg episode length = {ep_lens.mean():.1f}, max = {ep_lens.max()}")
print(f"  DT 在不同 target return 下的 eval reward 随 target 增长——符合 DT 理论。")
print(f"  但 target 远超数据集 max 时，DT 没见过这种 trajectory，eval 会停滞或下降。")


### 18.5.7 真实 DT 与本章简化的区别

真实的 Chen et al. 2021 DT 用 **GPT-style causal Transformer**，本章用 MLP。区别：

| 维度 | 真实 DT | 本章简化 |
|---|---|---|
| **backbone** | Causal Transformer（自回归） | MLP |
| **上下文长度** | 整条 trajectory（K 步历史） | 单步 $(R_t, s_t)$ |
| **能处理长程依赖？** | 是 | 否 |
| **何时简化合理？** | CartPole 这种短程任务 | 当任务"马尔可夫性强"时 |
| **何时简化失效？** | Atar、MuJoCo（长程依赖重要） | (本章不涉及) |

本章简化的目的是**清晰展示 return-conditioning 思想**——这是 DT 的灵魂。
真实 DT 的 Transformer 细节可以在 Chen 2021 原论文里看到。


## 18.6 三方对比实验

### 18.6.1 把四个方法（含 naive DQN）放在一起对比


In [ ]:
# 18.6.2 汇总对比：四个方法的 eval reward
methods = ['Naive DQN', 'CQL', 'IQL', f'DT (R={best_target})']
rewards = [naive_eval['mean'], cql_eval['mean'], iql_eval['mean'], dt_eval['mean']]
errors = [naive_eval['std'], cql_eval['std'], iql_eval['std'], dt_eval['std']]
colors = ['#d62728', '#2ca02c', '#9467bd', '#ff7f0e']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(methods, rewards, yerr=errors, color=colors, capsize=8,
              edgecolor='k', linewidth=1.2)
# 标注数值
for bar, r, e in zip(bars, rewards, errors):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + e + 3,
            f'{r:.1f}±{e:.1f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# 加参考线
ax.axhline(ep_lens.mean(), color='gray', linestyle='--', alpha=0.7,
           label=f'数据集 avg episode length = {ep_lens.mean():.1f}')
ax.axhline(ep_lens.max(), color='crimson', linestyle='--', alpha=0.7,
           label=f'数据集 max episode length = {ep_lens.max()} (best in data)')
ax.axhline(200, color='green', linestyle=':', alpha=0.7,
           label=f'max_steps = 200 (perfect)')
ax.set_ylabel('Eval reward (10 episodes)')
ax.set_title('Ch18 三方对比 + Naive DQN baseline\n(Offline 数据集上学到的策略质量)')
ax.set_ylim(0, 230)
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout(); plt.show()

print(f"{'方法':<12} {'eval reward':<18} {'Q max / value':<20} {'是否稳定'}")
print(f"{'-'*65}")
print(f"{'Naive DQN':<12} {naive_eval['mean']:<18.1f} Q max = {naive_history[-1]['q_max']:<10.1f} 不稳定（发散）")
print(f"{'CQL':<12} {cql_eval['mean']:<18.1f} Q max = {cql_history[-1]['q_all_max']:<10.1f} 稳定")
print(f"{'IQL':<12} {iql_eval['mean']:<18.1f} V mean = {iql_history[-1]['v_mean']:<10.1f} 稳定")
print(f"{'DT':<12} {dt_eval['mean']:<18.1f} (no Q) acc = {dt_history[-1]['action_acc']:<10.3f} 稳定")


In [ ]:
# 18.6.3 训练曲线对比：Q 值 / loss 的发散行为
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
steps_n = [h['step'] for h in naive_history]
steps_c = [h['step'] for h in cql_history]
steps_i = [h['step'] for h in iql_history]
steps_d = [h['step'] for h in dt_history]
ax.plot(steps_n, [h['q_max'] for h in naive_history], linewidth=2,
        color='#d62728', label='Naive DQN: Q max (incl OOD)')
ax.plot(steps_c, [h['q_all_max'] for h in cql_history], linewidth=2,
        color='#2ca02c', label='CQL: Q max (压低)')
ax.plot(steps_i, [h['v_mean'] for h in iql_history], linewidth=2,
        color='#9467bd', label='IQL: V mean (≈ max, 不外推)')
ax.axhline(true_max_q, color='gray', linestyle='--', linewidth=1.5,
           label=f'true max ≈ {true_max_q:.0f}')
ax.set_xlabel('step'); ax.set_ylabel('value')
ax.set_title('Q/V 值对比：Naive 发散，CQL/IQL 保守')
ax.legend(fontsize=8)

# 右图：归一化的 loss（让 4 条曲线可比较）
ax = axes[1]
def normalize(arr):
    arr = np.array(arr, dtype=float)
    return arr / max(arr.max(), 1e-8)

ax.plot(steps_n, normalize([h['loss'] for h in naive_history]),
        linewidth=2, color='#d62728', label='Naive DQN: Bellman loss')
ax.plot(steps_c, normalize([h['bellman_loss'] for h in cql_history]),
        linewidth=2, color='#2ca02c', label='CQL: Bellman loss')
ax.plot(steps_i, normalize([h['q_loss'] for h in iql_history]),
        linewidth=2, color='#9467bd', label='IQL: Q loss')
ax.plot(steps_d, normalize([h['loss'] for h in dt_history]),
        linewidth=2, color='#ff7f0e', label='DT: CE loss')
ax.set_xlabel('step'); ax.set_ylabel('normalized loss')
ax.set_title('四种方法的 loss 都在降，但 Naive 的 Q 仍爆炸')
ax.legend(fontsize=8)

plt.tight_layout(); plt.show()
print("关键洞察：从 loss 看四种方法都在收敛，但只有 Q/V 值监控才能发现 Naive 的发散。")
print("这就是 §18.2 distribution shift 的隐蔽性——它不体现在 loss 上，只体现在 Q 值 + 评估上。")


In [ ]:
# 18.6.4 算法对比表（一张大图）
fig, ax = plt.subplots(figsize=(13, 6))
ax.axis('off')
ax.set_title('Offline RL 四方法对比', fontsize=14, fontweight='bold', pad=20)

rows = [
    ['维度', '在线 DQN (Ch06)', 'Naive offline DQN', 'CQL', 'IQL', 'Decision Transformer'],
    ['数据来源', '在线 ε-greedy 采', '固定数据集', '固定数据集', '固定数据集', '固定数据集'],
    ['Bellman eq?', '是', '是', '是', '是', '否（纯监督）'],
    ['评估 OOD Q?', '是 (在线修正)', '是 (无法修正→爆炸)', '是 (保守化)', '否 (用 V 替代)', '否 (不用 Q)'],
    ['额外正则/网络', '无', '无', 'log-sum-exp 正则', 'V 网络 + expectile', 'return 输入'],
    ['策略提取', 'argmax Q', 'argmax Q', 'argmax Q', 'argmax Q 或 softmax', '条件化 R 生成 a'],
    ['数据集要求', '任意', '任意 (但会爆炸)', '任意', '任意', '需含高 return 轨迹'],
    ['本章 eval reward',
     f'(在线训可至 {ENV_MAX_STEPS})',
     f'{naive_eval["mean"]:.1f} (失败)',
     f'{cql_eval["mean"]:.1f}',
     f'{iql_eval["mean"]:.1f}',
     f'{dt_eval["mean"]:.1f}'],
]
colors_rows = [['#f0f0f0'] * 6]
for _ in rows[1:]:
    colors_rows.append(['#ffffff'] * 6)

table = ax.table(cellText=rows, cellColours=colors_rows,
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)

# 表头加粗 + 上色
for j in range(6):
    table[0, j].set_facecolor('#4a4a4a')
    table[0, j].set_text_props(color='white', fontweight='bold')
# Naive 列标红
for i in range(1, len(rows)):
    table[i, 2].set_facecolor('#ffcccc')

plt.tight_layout(); plt.show()


In [ ]:
# 18.6.5 总耗时统计
TOTAL_TIME = DATA_TIME + NAIVE_TIME + CQL_TIME + IQL_TIME + DT_TRAIN_TIME + DT_EVAL_TIME
print(f"Ch18 总耗时统计:")
print(f"  Offline 数据集收集:     {DATA_TIME:>6.1f}s")
print(f"  Naive DQN ({NAIVE_DQN_STEPS} steps):    {NAIVE_TIME:>6.1f}s")
print(f"  CQL ({CQL_STEPS} steps):              {CQL_TIME:>6.1f}s")
print(f"  IQL ({IQL_STEPS} steps):              {IQL_TIME:>6.1f}s")
print(f"  DT train ({DT_STEPS} steps):          {DT_TRAIN_TIME:>6.1f}s")
print(f"  DT eval (4 targets x 10 eps):  {DT_EVAL_TIME:>6.1f}s")
print(f"  {'─' * 40}")
print(f"  Total:                   {TOTAL_TIME:>6.1f}s ({TOTAL_TIME/60:.1f} min)")
print()
print("三种 offline RL 方法都在 ~30 秒内训完——远低于在线 RL 的耗时。")
print("这是 offline RL 的另一优势：训练成本可控（数据已固定）。")


## 18.7 Phase 4 终章 + 项目最终总结

### 18.7.1 Ch18 核心收获

| 概念 | 一句话总结 | 出处 |
|---|---|---|
| **Offline RL 设定** | 只用历史数据，不再与环境交互 | §18.1 |
| **Distribution shift** | 学到的策略可能去 OOD 区域，Q 外推不可靠 | §18.2 |
| **Naive DQN 在 offline 上发散** | max 的正向偏置累积爆炸 | §18.2 |
| **CQL** | Q-learning + log-sum-exp 正则压低 OOD Q | §18.3 |
| **IQL** | 用 V（expectile）替代 max，完全不评估 OOD | §18.4 |
| **Expectile regression** | τ-expectile 是 τ 偏尾的 squared-loss 解 | §18.4.2 |
| **Decision Transformer** | RL = return-conditioned 监督学习 | §18.5 |
| **Return-to-go** | $R_t = \sum_{t' \ge t} \gamma^{t'-t} r_{t'}$ | §18.5.3 |
| **DT 的限制** | 不能超越数据集（只能复制成功行为） | §18.5.4 |

### 18.7.2 关键公式速查

| 公式 | 含义 | 出处 |
|---|---|---|
| $\mathcal{L}_{CQL} = \mathcal{L}_{DQN} + \alpha[\log\sum_a e^Q - \mathbb{E}_\mathcal{D}[Q]]$ | CQL loss | §18.3 |
| $\rho_\tau(u) = \|\tau - \mathbb{1}(u<0)\| \cdot u^2$ | Expectile loss | §18.4.2 |
| $\mathcal{L}_V = \mathbb{E}[\rho_\tau(Q(s, a_{data}) - V(s))]$ | IQL V loss | §18.4.4 |
| $\mathcal{L}_Q = \mathbb{E}[(r + \gamma V(s') - Q)^2]$ | IQL Q loss（**无 max**） | §18.4.4 |
| $\mathcal{L}_{DT} = -\mathbb{E}[\log \pi(a \| R, s)]$ | DT loss（监督 CE） | §18.5.2 |

### 18.7.3 与 Ch06 / Ch09 / Ch13 的关系（项目闭环）

```
数据来源维度:
  on-policy         off-policy          offline           "no-policy"
  (在线采样)         (在线采样+buffer)    (固定数据集)        (没数据，纯监督)
  ────────────────────────────────────────────────────────────────>
  Ch07-09 PPO       Ch05-06 Q-learning  Ch18 CQL/IQL      Ch18 DT
  Ch12 RLHF-PPO     Ch13 GRPO           (本章)            (本章)
  (rollout 必须)    (rollout 必须)      (rollout 不允许)   (rollout 不需要)
```

- **Ch06 DQN** → Ch18 Naive offline DQN（直接喂固定数据）→ **失败**
- **Ch06 + 正则** → Ch18 CQL（成功）
- **Ch06 + 改架构** → Ch18 IQL（成功）
- **彻底放弃 Ch06 思路** → Ch18 DT（成功）

### 18.7.4 与 LLM 对齐的关系（为什么 RLStudy 把 Offline RL 放在终章）

Offline RL 不是 LLM 对齐的主流（主流仍是 PPO/GRPO 在线 RL），但它在两个方向上
**与 LLM 高度相关**：

1. **历史对话 log 的利用**：LLM 公司有海量用户对话 log，直接用 RL 从中学习
   （而非在线 rollout）能大幅降本——这是 Offline RL 在 LLM 上的潜在应用。

2. **Decision Transformer 路线 → 序列建模回归**：DT 的思想"把 RL 看成 sequence
   modeling" 与 LLM 的核心能力（自回归生成）**天然契合**。一些研究（如 Trajectory
   Transformer、Agent Transformer）已经把 DT 思路用于 LLM agent。

3. **DPO（Ch14）也是"Offline"的一种**：DPO 用偏好数据集直接训，不在线 rollout。
   可以把 DPO 看作"分类版 offline RL"——它和 CQL/IQL 共享"从固定数据学策略"的本质，
   只是用监督 loss 替代 Bellman loss。


### 18.7.5 Phase 4 全回顾（三章）

**Phase 4**：Ch15 §15.6.3 列的 7 个开放方向，逐个展开。

| 章 | 主题 | 兑现的开放方向 | 核心贡献 |
|---|---|---|---|
| **Ch16** | Process Reward Model (PRM) | 方向 5 | OpenAI o1 的核心：步骤级 reward |
| **Ch17** | Self-Play + Constitutional AI / RLAIF | 方向 2, 3 | 减少 RLHF 对人类标注的依赖 |
| **Ch18** | Offline RL（CQL / IQL / DT） | **方向 4** | 减少对在线交互的依赖 |

Phase 4 三章的共同主题：

> **让 RL 在真实场景中更实用**——减少对昂贵资源（人类标注 / 在线交互）的依赖。


### 18.7.6 整个 RLStudy 项目最终回顾（Ch00-Ch18 共 19 章）

```
RLStudy 全书路径（19 章）:

Phase 1: 经典 RL 基础 (Ch00-05)
  Ch00  环境搭建 + RL 全景         工具链、自研环境
  Ch01  多臂老虎机                 探索-利用、ε-greedy、UCB
  Ch02  MDP + 贝尔曼方程           状态价值、动作价值、γ
  Ch03  动态规划                   Policy/Value Iteration
  Ch04  TD 学习                    MC vs TD、TD(λ)
  Ch05  Q-learning / SARSA         off-policy vs on-policy

Phase 2: 深度 RL + 策略梯度 (Ch06-09)
  Ch06  DQN + 函数逼近             神经网络 + replay + target net
  Ch07  策略梯度定理               REINFORCE、baseline
  Ch08  Actor-Critic + GAE         A2C、advantage 分解
  Ch09  TRPO + PPO                 trust region、clip

Phase 3: LLM + RLHF + GRPO (Ch10-15)
  Ch10  TinyGPT 从零搭             self-attention、Transformer
  Ch11  Reward Modeling            Bradley-Terry、偏好学习
  Ch12  RLHF-PPO (InstructGPT)     SFT + RM + PPO 完整 pipeline
  Ch13  GRPO (DeepSeek-R1)         group baseline、无 critic
  Ch14  DPO / KTO                  不用 RL 的对齐
  Ch15  终局项目 + 开放方向         R1-style 训练、7 个开放方向

Phase 4: 研究前沿（Ch15 §15.6.3 逐个展开）(Ch16-18)
  Ch16  PRM (Process Reward Model) o1 风格步骤级 reward
  Ch17  Self-Play + CAI / RLAIF    无人工标注的对齐
  Ch18  Offline RL                 用历史数据学策略        ← 本章（终章）
```

### 18.7.7 各章节核心算法对照

| 章 | 核心算法 | 类别 |
|---|---|---|
| Ch01 | ε-greedy / UCB | bandit |
| Ch02 | Bellman equation | MDP |
| Ch03 | Value/Policy Iteration | DP |
| Ch04 | TD(λ) | TD |
| Ch05 | Q-learning / SARSA | tabular RL |
| Ch06 | DQN / Double / Dueling | deep RL |
| Ch07 | REINFORCE | policy gradient |
| Ch08 | A2C + GAE | actor-critic |
| Ch09 | PPO-Clip | trust region |
| Ch10 | Transformer | LLM 基础 |
| Ch11 | Bradley-Terry RM | reward model |
| Ch12 | RLHF (SFT+RM+PPO) | LLM 对齐 |
| Ch13 | GRPO | LLM 对齐（无 critic） |
| Ch14 | DPO / KTO | offline LLM 对齐 |
| Ch15 | Capstone | 项目整合 |
| Ch16 | PRM | 步骤级 reward |
| Ch17 | SPIN / RLAIF | self-play 对齐 |
| **Ch18** | **CQL / IQL / DT** | **offline RL** |


### 18.7.8 读者下一步路径

读完 RLStudy 19 章后，你已经掌握了：

1. **经典 RL 全套**：从 bandit 到 PPO，能读懂任何 RL 教材（Sutton & Barto、Sergey Levine CS285）
2. **LLM 对齐主流算法**：RLHF-PPO、GRPO、DPO、KTO 都能从零实现
3. **研究前沿**：PRM、Self-Play、Offline RL——能读懂 2024-2025 的前沿论文

接下来推荐的深入方向：

| 方向 | 入门材料 |
|---|---|
| **大规模 RLHF 工程实现** | TRL 库、DeepSpeed-Chat、OpenRLHF 源码 |
| **多模态 RLHF** | LLaVA-RLHF、RLHF-V 论文 |
| **RLHF 替代：迭代 DPO** | Self-Rewarding LM、Iterative DPO |
| **Agent RL** | ToolFormer、ReAct、Reflexion |
| **World Models** | Dreamer V3、JEPA |
| **可解释 RL** | Attention 解释、Mechanistic interpretability |
| **Offline RL 深入** | Sergey Levine CS267、D4RL benchmark |
| **Diffusion + RL** | Diffusion Q-Learning、Decision Diffuser |

### 18.7.9 Ch15 §15.6.3 开放方向 4 的兑现

> **方向 4**：Offline RL / Decision Transformer

本章完整兑现：

| 维度 | 体现 |
|---|---|
| **理论** | §18.1-18.5 完整推导 CQL/IQL/DT |
| **代码** | `utils/offline_rl.py` 实现 6 个核心组件 |
| **实验** | §18.6 四方法对比（含失败 baseline） |
| **测试** | `tests/test_offline_rl.py` 23 个冒烟测试 |
| **结论** | 三种方法都能从 offline 数据学到合理策略；naive DQN 失败 |


In [ ]:
# Ch18 完成总结 + 整个 RLStudy 项目最终交付
print('=' * 72)
print('Ch18 Offline RL（CQL / IQL / Decision Transformer）完成')
print('  (Phase 4 第三章、整个 RLStudy 项目终章)')
print('  Ch15 §15.6.3 开放方向 4: Offline RL')
print('=' * 72)
print()
print('本章交付:')
print('  - utils/offline_rl.py')
print('      collect_offline_dataset / OfflineDataset  (数据收集 + 容器)')
print('      offline_dqn_update_step                  (naive baseline)')
print('      cql_loss / CQLTrainer                    (Conservative Q-Learning)')
print('      expectile_loss / IQLTrainer              (Implicit Q-Learning)')
print('      DecisionTransformer / DTTrainer          (RL = Sequence Modeling)')
print('      dt_rollout / evaluate_policy             (评估工具)')
print('  - notebooks/ch18_offline_rl.ipynb:           本章（项目终章）')
print('  - tests/test_offline_rl.py:                  23 个冒烟测试')
print()
print('四方法实验结果（在 800 episode offline 数据集上）:')
print(f"  {'方法':<14} {'eval reward':<18} {'Q max':<14} {'状态'}")
print(f"  {'-'*60}")
print(f"  {'Naive DQN':<14} {naive_eval['mean']:<18.1f} "
      f"Q max={naive_history[-1]['q_max']:<7.1f} ❌ 发散")
print(f"  {'CQL':<14} {cql_eval['mean']:<18.1f} "
      f"Q max={cql_history[-1]['q_all_max']:<7.1f} ✅ 稳定")
print(f"  {'IQL':<14} {iql_eval['mean']:<18.1f} "
      f"V mean={iql_history[-1]['v_mean']:<7.1f} ✅ 稳定")
print(f"  {'DT':<14} {dt_eval['mean']:<18.1f} "
      f"acc={dt_history[-1]['action_acc']:<7.3f} ✅ 稳定")
print()
print(f'总耗时（notebook）: {TOTAL_TIME:.1f}s ({TOTAL_TIME/60:.1f} min)')
print()
print('=' * 72)
print('RLStudy 项目最终交付（Ch00-Ch18 共 19 章）')
print('=' * 72)
print()
print('Phase 1: 经典 RL 基础 (Ch00-05)         ✅ 全部交付')
print('  Ch00 环境搭建 + RL 全景')
print('  Ch01 多臂老虎机          (bandit, ε-greedy, UCB)')
print('  Ch02 MDP + 贝尔曼方程    (V, Q, γ)')
print('  Ch03 动态规划            (Policy/Value Iteration)')
print('  Ch04 TD 学习             (MC vs TD, TD(λ))')
print('  Ch05 Q-learning / SARSA  (off vs on policy)')
print()
print('Phase 2: 深度 RL + 策略梯度 (Ch06-09)    ✅ 全部交付')
print('  Ch06 DQN                 (DQN/Double/Dueling)')
print('  Ch07 策略梯度定理        (REINFORCE)')
print('  Ch08 Actor-Critic + GAE  (A2C, advantage 分解)')
print('  Ch09 TRPO + PPO          (PPO-Clip)')
print()
print('Phase 3: LLM + RLHF + GRPO (Ch10-15)     ✅ 全部交付')
print('  Ch10 TinyGPT             (Transformer)')
print('  Ch11 Reward Modeling     (Bradley-Terry)')
print('  Ch12 RLHF-PPO            (InstructGPT pipeline)')
print('  Ch13 GRPO                (DeepSeek-R1, no critic)')
print('  Ch14 DPO / KTO           (offline alignment)')
print('  Ch15 终局项目            (capstone + 7 个开放方向)')
print()
print('Phase 4: 研究前沿 (Ch16-18)               ✅ 全部交付')
print('  Ch16 PRM                 (Process Reward Model, o1 核心)')
print('  Ch17 Self-Play + CAI     (RLAIF, 无人工标注)')
print('  Ch18 Offline RL          (CQL/IQL/DT, 项目终章)  ← 本章')
print()
print('项目完成度: 100%')
print('  章节:      19 章（Ch00-Ch18）')
print('  笔记本:    19 个独立可跑的 ipynb（每个 < 10 分钟）')
print('  测试:      157 个冒烟测试全部通过')
print('  代码:      ~15 个 utils 模块 + 7 个 rlenvs 环境')
print()
print('核心学习路径（推荐）:')
print('  零基础 → Ch00-09 (Phase 1+2, 经典+深度 RL)')
print('        → Ch10-13 (Phase 3 主线, LLM+RLHF+GRPO)')
print('        → Ch14-15 (DPO 替代方案 + capstone)')
print('        → Ch16-18 (Phase 4 研究前沿)')
print()
print('=' * 72)
print('🎉 RLStudy 项目交付完成。感谢读到这里——你已掌握从零到 GRPO 训 LLM 的')
print('   完整数学与工程基础。下一步是去看真实论文、改真实代码、跑真实实验。')
print('=' * 72)
